# Práctica 3 · Modelos de decisión
## La bandeja de entrada de Vallatech

**12 correos simulados → 12 fichas para el equipo organizador**

Cada ficha indica quién puede atender el correo, si activaría un aviso inmediato y si hay información suficiente para actuar.

## 1 · Cargar Laya

En Colab, elige **GPU T4** si está disponible y ejecuta de arriba abajo. También funciona en CPU. Si ya importaste las librerías en esta sesión, reinicia el entorno después de instalarlas.

Usamos el SDK, los pesos multilingües y PyTorch. Los pesos se descargan una vez, sin clave de API.

**¿En qué hardware estás ejecutando la inferencia?** La celda de carga muestra si el entorno ha elegido GPU o CPU.

In [ ]:
%pip -q install "laya==0.3.20" "transformers==5.20.2" "huggingface-hub==1.32.0" pandas matplotlib

In [ ]:
import json
import os
import time
from importlib.metadata import version
from pathlib import Path

import laya
import matplotlib.pyplot as plt
import pandas as pd
import torch
from huggingface_hub import snapshot_download
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
LAYA_ID = "convaiinnovations/laya"
LAYA_REV = "7b928d828b7b0e022f929d9bd2e44165aa270148"

snapshot = snapshot_download(
    LAYA_ID, revision=LAYA_REV,
    allow_patterns=["multilingual/rl_agent_config.json", "multilingual/model.safetensors",
                    "multilingual/tokenizer/*", "multilingual/encoder/*"],
)
agent = laya.load(str(Path(snapshot) / "multilingual"), device=DEVICE, fast=False)
# El SDK puede recurrir a CPU si no hay memoria suficiente en la GPU.
DEVICE = agent.device.type
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
hardware = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
print(f"Laya {version('laya')} · {hardware}")

## 2 · Del correo a la ficha

**¿Qué devuelve Laya y qué decide nuestro código?** Ejecuta el ejemplo y compara los valores originales con la ficha que consumiría la bandeja.

El **estado** es el asunto y el cuerpo del correo. Las **preguntas tipadas** y sus **criterios** forman la política que se aplica a toda la bandeja.

| Campo | Tipo | Salida |
|---|---|---|
| Categoría | `choice` | ponentes / participantes / logistica / patrocinio / otros |
| Avisar por Telegram | `noul` | Sí si P(sí) ≥ 0,5 |
| Información disponible | `score` | 0 falta el dato básico · 1 falta confirmar algo · 2 se puede actuar |

Laya devuelve un valor esperado para `score`. Lo redondeamos al nivel más cercano (en caso de empate, hacia arriba) para compararlo con la etiqueta de referencia. El umbral de aviso pertenece a la política simulada del taller.

In [ ]:
UMBRAL_AVISO = 0.5

QUESTIONS = {
    "categoria": {
        "type": "choice",
        "instructions": "¿Cuál es el asunto principal del correo?",
        "criteria": {
            "ponentes": "Una charla o un taller, su ponente o su material.",
            "participantes": "Entradas, acreditaciones o preguntas de asistentes.",
            "logistica": "La sede, sus salas, el equipamiento común o el montaje.",
            "patrocinio": "Una empresa y su aportación, stand o visibilidad como patrocinadora.",
            "otros": "Prensa, consultas generales y otros asuntos.",
        },
    },
    "avisar_telegram": {
        "type": "noul",
        "instructions": "¿El correo describe un problema confirmado que requiere avisar ahora al equipo?",
        "criteria": {
            "true": "Cancelación de última hora, problema con la sede o bloqueo de una sesión que empieza pronto.",
            "false": "Consulta rutinaria, planificación o asunto que puede esperar.",
        },
        "labels": {"true": "A", "false": "B"},
    },
    "informacion": {
        "type": "score",
        "instructions": "¿Cuánta información aporta el correo para resolver el asunto o responder a la consulta?",
        "criteria": [
            "Falta el dato básico para identificar el asunto.",
            "Se entiende el asunto, pero falta confirmar o pedir un dato clave para resolverlo.",
            "El asunto está identificado y ya se puede responder o actuar con los datos disponibles.",
        ],
    },
}

def ficha_laya(resultado):
    respuestas = resultado["answers"]
    return {
        "categoria": respuestas["categoria"]["choice"],
        "avisar_telegram": float(respuestas["avisar_telegram"]["noul"]) >= UMBRAL_AVISO,
        "informacion": int(float(respuestas["informacion"]["score"]) + 0.5),
    }

ASUNTO_DEMO = "No llegaré al taller de las 12:00"
CUERPO_DEMO = (
    "Soy la ponente del taller de Python de las 12:00 de hoy. "
    "Mi tren se ha cancelado y no llegaré a tiempo. "
    "En diez minutos os confirmo si puedo impartirlo en remoto."
)
correo_demo = f"Asunto: {ASUNTO_DEMO}\n\n{CUERPO_DEMO}"
from html import escape
from IPython.display import HTML

resultado_demo = agent.predict(correo_demo, QUESTIONS)
ficha_demo = ficha_laya(resultado_demo)
respuestas_demo = resultado_demo["answers"]

print(correo_demo)
print(f"P(sí) de aviso: {respuestas_demo['avisar_telegram']['noul']:.3f} · umbral: {UMBRAL_AVISO}")
print(f"Información: {respuestas_demo['informacion']['score']:.3f} → {ficha_demo['informacion']}")
print(json.dumps(ficha_demo, ensure_ascii=False, indent=2))

crudo = json.dumps(resultado_demo, ensure_ascii=False, indent=2, default=str)
display(HTML(f"<details><summary>Respuesta cruda de Laya</summary><pre>{escape(crudo)}</pre></details>"))

## 3 · Procesar la bandeja con Laya

**¿Qué asuntos aparecerían en Telegram?** Antes de ejecutar el lote, piensa cuáles requieren interrumpir al equipo.

`predict_batch` aplica la misma política a todos los correos. Python prepara una ficha por correo y muestra una vista previa de los avisos de Telegram. Los asuntos y cuerpos son simulados y la vista previa se queda en Colab.

In [ ]:
CASOS = [
    (ASUNTO_DEMO, CUERPO_DEMO, "ponentes", True, 1),
    ("¿Puedo ir en bici?", "Soy asistente. ¿Habrá un aparcabicis en la entrada del evento?", "participantes", False, 2),
    ("La sede no abre", "La puerta principal de la sede está cerrada y la apertura es en 15 minutos. El conserje está en la entrada lateral y puede abrirla si le damos permiso.", "logistica", True, 2),
    ("Logo para la web", "Somos patrocinadores. ¿Qué tamaño debe tener nuestro logo para aparecer en la página del evento?", "patrocinio", False, 2),
    ("Una duda", "Hola, quería preguntar una cosa. ¿Me podéis ayudar?", "otros", False, 0),
    ("No funciona el proyector", "El taller de las 10:00 empieza en diez minutos. El proyector de la sala B no enciende. El técnico está en la puerta con un proyector de repuesto listo para instalar.", "logistica", True, 2),
    ("Cambio de nombre en una entrada", "Soy asistente y necesito corregir el nombre de mi acreditación. Os paso el número de entrada en cuanto lo encuentre.", "participantes", False, 1),
    ("Material de mi charla", "Soy ponente de la charla de APIs del viernes a las 16:00. ¿Aceptáis las diapositivas en PDF?", "ponentes", False, 2),
    ("Ubicación del stand", "Somos patrocinadores y queremos cambiar nuestro stand. Mañana confirmamos cuánto espacio necesitamos.", "patrocinio", False, 1),
    ("Contacto de prensa", "Escribo para una revista local. ¿Quién puede atender una entrevista sobre el evento la semana que viene?", "otros", False, 2),
    ("Montaje de la sala", "Tengo una consulta sobre el montaje, pero todavía no sé de qué sala ni de qué equipo se trata.", "logistica", False, 0),
    ("Cancelo mi charla de hoy", "No podré impartir la charla de diseño de APIs de las 17:00, que empieza en media hora. El equipo suplente ya está en la sala y confirma que puede cubrir ese hueco.", "ponentes", True, 2),
]
CORREOS = [f"Asunto: {asunto}\n\n{cuerpo}" for asunto, cuerpo, *_ in CASOS]
ESPERADAS = [
    {"categoria": categoria, "avisar_telegram": aviso, "informacion": nivel}
    for _, _, categoria, aviso, nivel in CASOS
]

def ejecutar_laya(correos):
    return [ficha_laya(r) for r in agent.predict_batch(correos, QUESTIONS)]

fichas = ejecutar_laya(CORREOS)
display(pd.DataFrame(fichas).assign(asunto=[caso[0] for caso in CASOS]).set_index("asunto"))

print("Telegram · vista previa simulada")
avisos = [
    f"[Vallatech · {ficha['categoria']}] {caso[0]}"
    for caso, ficha in zip(CASOS, fichas, strict=True)
    if ficha["avisar_telegram"]
]
print("\n".join(avisos) or "Ningún correo activa un aviso.")

## 4 · Misma bandeja con un LLM generativo

**¿Puede Qwen entregar una ficha que el programa lea directamente?** La política y los campos son los mismos. Más abajo podrás ver el texto original y comprobar si respeta el formato.

Qwen recibe los mismos correos y criterios. Le pedimos una ficha JSON por correo, con los mismos tres campos. Ambos modelos procesan la bandeja en lote.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN_ID = "Qwen/Qwen2.5-0.5B-Instruct"
QWEN_REV = "7ae557604adf67be50417f59c2c2f167def9a775"
MAX_NEW_TOKENS = 64
tokenizer = AutoTokenizer.from_pretrained(QWEN_ID, revision=QWEN_REV, padding_side="left")
llm = AutoModelForCausalLM.from_pretrained(
    QWEN_ID, revision=QWEN_REV, dtype=DTYPE, use_safetensors=True,
).to(DEVICE).eval()
def comprobar_dispositivo():
    if next(agent.model.parameters()).device != llm.device:
        raise RuntimeError("Los modelos deben ejecutarse en el mismo dispositivo. Reinicia el entorno y vuelve a cargarlos.")

comprobar_dispositivo()
print(f"Qwen cargado · {llm.device}")

In [ ]:
def sincronizar():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

def prompt_qwen():
    reglas = []
    for campo, pregunta in QUESTIONS.items():
        criterios = pregunta["criteria"]
        opciones = criterios.items() if isinstance(criterios, dict) else enumerate(criterios)
        detalle = "; ".join(f"{opcion} = {criterio}" for opcion, criterio in opciones)
        reglas.append(f"{campo}: {pregunta['instructions']} {detalle}")
    return (
        "Clasifica un correo de Vallatech con estas reglas:\n" + "\n".join(reglas)
        + '\nDevuelve solo un objeto JSON con exactamente tres claves: "categoria" '
          'como texto con una de las opciones, "avisar_telegram" como booleano JSON '
          '(true o false) e "informacion" como entero (0, 1 o 2). '
          'Usa solo el asunto y el cuerpo del correo.'
    )

def leer_json(texto):
    try:
        return json.loads(texto)
    except ValueError:
        return None

@torch.inference_mode()
def ejecutar_qwen(correos, streamer=None):
    comprobar_dispositivo()
    instrucciones = prompt_qwen()
    chats = [
        [{"role": "system", "content": instrucciones}, {"role": "user", "content": correo}]
        for correo in correos
    ]
    entrada = tokenizer.apply_chat_template(
        chats, tokenize=True, add_generation_prompt=True, padding=True,
        return_dict=True, return_tensors="pt",
    ).to(DEVICE)
    salida = llm.generate(
        **entrada, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id, streamer=streamer,
    )
    texto = tokenizer.batch_decode(
        salida[:, entrada["input_ids"].shape[1]:], skip_special_tokens=True,
    )
    return [leer_json(t) for t in texto], texto

### Demo · Cómo llega la respuesta de cada modelo

**¿Cuándo ves la primera parte de la respuesta y cuándo tienes una ficha completa?** Elige un correo del 1 al 12. Laya muestra la ficha al terminar y Qwen actualiza su salida con los tokens que va recibiendo. Cada llamada tiene su propio reloj. El primer token puede no mostrar texto y el total de Qwen incluye las actualizaciones del panel. El bloque de medición mide el lote con la interfaz fuera del reloj.

In [ ]:
# @title Ver un correo en directo
numero_correo = 1 # @param {type:"integer"}
from html import escape
from IPython.display import HTML
from transformers.generation.streamers import BaseStreamer

if not 1 <= numero_correo <= len(CORREOS):
    raise ValueError(f"Elige un correo entre 1 y {len(CORREOS)}.")
correo = CORREOS[numero_correo - 1]

# Calienta una llamada de cada modelo antes de iniciar la demo.
ejecutar_laya([correo])
ejecutar_qwen([correo])

def panel(laya_texto, laya_estado, qwen_texto, qwen_estado):
    def tarjeta(titulo, texto, estado):
        return (
            '<div style="border:1px solid #8888;padding:1em;min-width:0">'
            f'<strong>{escape(titulo)}</strong><p>{escape(estado)}</p>'
            f'<pre style="white-space:pre-wrap">{escape(texto)}</pre></div>'
        )

    return HTML(
        f'<pre style="white-space:pre-wrap">{escape(correo)}</pre>'
        '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(240px,1fr));gap:1em">'
        + tarjeta("Laya · ficha completa", laya_texto, laya_estado)
        + tarjeta("Qwen · generación en directo", qwen_texto, qwen_estado)
        + '</div>'
    )

class TokensEnDirecto(BaseStreamer):
    def __init__(self, actualizar):
        self.actualizar = actualizar
        self.es_entrada = True
        self.tokens = []
        self.primer_token_ms = None

    def put(self, value):
        if self.es_entrada:
            self.es_entrada = False
            return
        for token in value.reshape(-1).tolist():
            if self.primer_token_ms is None:
                self.primer_token_ms = (time.perf_counter() - self.inicio) * 1000
            self.tokens.append(token)
            texto = tokenizer.decode(self.tokens, skip_special_tokens=True)
            self.actualizar(texto, self.estado())

    def estado(self):
        if self.primer_token_ms is None:
            return "0 tokens recibidos"
        return f"{len(self.tokens)} tokens · primero en {self.primer_token_ms:.1f} ms"

    def end(self):
        pass

vista = display(panel("", "Procesando…", "", "En espera"), display_id=True)
sincronizar()
inicio_laya = time.perf_counter()
ficha = ejecutar_laya([correo])[0]
sincronizar()
laya_ms = (time.perf_counter() - inicio_laya) * 1000
laya_texto = json.dumps(ficha, ensure_ascii=False, indent=2)
laya_estado = f"Completa en {laya_ms:.1f} ms · 0 tokens generados"

def actualizar_qwen(texto, estado):
    vista.update(panel(laya_texto, laya_estado, texto, estado))

actualizar_qwen("", "Esperando el primer token…")
stream = TokensEnDirecto(actualizar_qwen)
sincronizar()
stream.inicio = time.perf_counter()
_, textos = ejecutar_qwen([correo], streamer=stream)
sincronizar()
qwen_ms = (time.perf_counter() - stream.inicio) * 1000
actualizar_qwen(textos[0], f"{stream.estado()} · completa en {qwen_ms:.1f} ms")

### Demo alternativa · Cómo llega la respuesta de todo el lote

**¿Aparecen las doce fichas juntas o se van escribiendo en paralelo?** Laya devuelve el lote completo. Qwen genera las doce respuestas en paralelo y el panel se actualiza después de cada paso de generación. Los pasos incluyen el relleno de las secuencias que ya han terminado. Cada modelo tiene su propio reloj y el de Qwen incluye el panel.

In [ ]:
# @title Ver los 12 correos en directo
from html import escape
from IPython.display import HTML
from transformers.generation.streamers import BaseStreamer

# El calentamiento queda fuera de los dos relojes.
ejecutar_laya(CORREOS)
ejecutar_qwen(CORREOS)

asuntos = [caso[0] for caso in CASOS]
vacios = [""] * len(CORREOS)

def panel_lote(laya_fichas, laya_estado, qwen_textos, qwen_estado):
    filas = []
    for asunto, ficha, texto in zip(asuntos, laya_fichas, qwen_textos, strict=True):
        filas.append(
            f"<tr><td>{escape(asunto)}</td>"
            f'<td><code style="white-space:pre-wrap">{escape(ficha)}</code></td>'
            f'<td><code style="white-space:pre-wrap">{escape(texto)}</code></td></tr>'
        )
    return HTML(
        '<table style="table-layout:fixed;width:100%;overflow-wrap:anywhere">'
        '<thead><tr><th>Correo</th>'
        f"<th>Laya · {escape(laya_estado)}</th><th>Qwen · {escape(qwen_estado)}</th>"
        "</tr></thead><tbody>" + "".join(filas) + "</tbody></table>"
    )

class LoteEnDirecto(BaseStreamer):
    def __init__(self, actualizar):
        self.actualizar = actualizar
        self.es_entrada = True
        self.pasos = 0
        self.primer_paso_ms = None
        self.tokens = [[] for _ in CORREOS]

    def put(self, value):
        if self.es_entrada:
            self.es_entrada = False
            return
        if self.primer_paso_ms is None:
            self.primer_paso_ms = (time.perf_counter() - self.inicio) * 1000
        # Cada paso aporta una posición por correo. Las secuencias terminadas reciben padding.
        for ids, token in zip(self.tokens, value.reshape(-1).tolist(), strict=True):
            ids.append(token)
        self.pasos += 1
        textos = tokenizer.batch_decode(self.tokens, skip_special_tokens=True)
        self.actualizar(textos, self.estado())

    def estado(self):
        if self.primer_paso_ms is None:
            return "Sin pasos de generación"
        return f"{self.pasos} pasos · primero en {self.primer_paso_ms:.1f} ms"

    def end(self):
        pass

vista_lote = display(panel_lote(vacios, "Procesando…", vacios, "En espera"), display_id=True)
sincronizar()
inicio_lote = time.perf_counter()
fichas_lote = ejecutar_laya(CORREOS)
sincronizar()
laya_lote_ms = (time.perf_counter() - inicio_lote) * 1000
laya_fichas = [json.dumps(ficha, ensure_ascii=False) for ficha in fichas_lote]
laya_estado_lote = f"{len(laya_fichas)} fichas en {laya_lote_ms:.1f} ms"

def actualizar_lote(textos, estado):
    vista_lote.update(panel_lote(laya_fichas, laya_estado_lote, textos, estado))

actualizar_lote(vacios, "Esperando el primer paso…")
stream_lote = LoteEnDirecto(actualizar_lote)
sincronizar()
stream_lote.inicio = time.perf_counter()
_, textos_lote = ejecutar_qwen(CORREOS, streamer=stream_lote)
sincronizar()
qwen_lote_ms = (time.perf_counter() - stream_lote.inicio) * 1000
actualizar_lote(textos_lote, f"{stream_lote.estado()} · lote en {qwen_lote_ms:.1f} ms")

## 5 · Medir el lote completo

**¿Cuánto tarda cada modelo en procesar los doce correos?** Compara las medianas del lote y los correos por segundo.

Tres pasadas después de calentar ambos modelos. Mismo hardware y mismos correos, alternando el orden. El reloj incluye preparar la entrada, inferir y obtener las fichas. Descarga, carga, calentamiento y vista previa quedan fuera. Qwen tiene un presupuesto de hasta 64 tokens nuevos por correo; revisa también los aciertos y las salidas rechazadas.

In [ ]:
def medir(ejecutar):
    sincronizar()
    inicio = time.perf_counter()
    salida = ejecutar(CORREOS)
    sincronizar()
    return salida, (time.perf_counter() - inicio) * 1000

ejecutar_laya(CORREOS)
ejecutar_qwen(CORREOS)
comprobar_dispositivo()

ejecutores = {"Laya": ejecutar_laya, "Qwen": ejecutar_qwen}
medidas = []
salidas = {}
for pasada in range(1, 4):
    orden = list(ejecutores)
    if pasada % 2 == 0:
        orden.reverse()
    for modelo in orden:
        salida, ms = medir(ejecutores[modelo])
        comprobar_dispositivo()
        salidas[modelo] = salida
        medidas.append({"modelo": modelo, "pasada": pasada, "ms": ms})

tiempos = pd.DataFrame(medidas)
medianas = tiempos.groupby("modelo")["ms"].median()
fig, ax = plt.subplots(figsize=(9, 3))
barras = ax.barh(medianas.index, medianas, color=["#365797", "#5c6576"])
ax.bar_label(barras, labels=[f"{ms:,.1f} ms · {len(CORREOS) * 1000 / ms:,.1f} correos/s" for ms in medianas], padding=6, fontsize=12)
ax.set_xlim(0, medianas.max() * 1.55)
ax.set_title(f"Tanda de {len(CORREOS)} correos · mediana de 3 pasadas · {hardware}", fontsize=14)
ax.set_xlabel("Tiempo de la tanda completa (ms)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6 · Revisar las fichas

**¿Qué modelo entrega más fichas utilizables? ¿Qué avisos se perderían?** Compara la tabla de resultados y abre el primer ejemplo rechazado si lo hay.

**Formato válido** exige un objeto JSON con los tres campos y valores admitidos. **Ficha correcta** exige acertar los tres campos. Una ficha inválida cuenta como fallo. La revisión usa la última pasada y las etiquetas manuales.

Los avisos omitidos e innecesarios se cuentan entre las fichas válidas. Los correos urgentes con ficha inválida se muestran aparte.

In [ ]:
def formato_valido(ficha):
    if not isinstance(ficha, dict) or set(ficha) != set(QUESTIONS):
        return False
    return (
        isinstance(ficha["categoria"], str)
        and ficha["categoria"] in QUESTIONS["categoria"]["criteria"]
        and type(ficha["avisar_telegram"]) is bool
        and type(ficha["informacion"]) is int
        and 0 <= ficha["informacion"] < len(QUESTIONS["informacion"]["criteria"])
    )

qwen_fichas, qwen_textos = salidas["Qwen"]
revision = []
for modelo, predicciones in [("Laya", salidas["Laya"]), ("Qwen", qwen_fichas)]:
    for i, (esperada, ficha) in enumerate(zip(ESPERADAS, predicciones, strict=True)):
        valida = formato_valido(ficha)
        revision.append({
            "modelo": modelo, "correo": CORREOS[i],
            "esperada": json.dumps(esperada, ensure_ascii=False),
            "obtenida": qwen_textos[i] if modelo == "Qwen" else json.dumps(ficha, ensure_ascii=False),
            "formato válido": valida, "ficha correcta": valida and ficha == esperada,
            **{f"acierto {campo}": valida and ficha[campo] == valor for campo, valor in esperada.items()},
        })

revision = pd.DataFrame(revision)
columnas = ["formato válido", "ficha correcta", *[f"acierto {campo}" for campo in QUESTIONS]]
totales = revision.groupby("modelo")[columnas].sum()
resumen = totales.map(lambda n: f"{n}/{len(CORREOS)}")
resumen.insert(0, "mediana lote (ms)", medianas.round(1))
display(resumen)

avisos_revision = []
for modelo, predicciones in [("Laya", salidas["Laya"]), ("Qwen", qwen_fichas)]:
    validas = [
        (esperada["avisar_telegram"], ficha["avisar_telegram"])
        for esperada, ficha in zip(ESPERADAS, predicciones, strict=True)
        if formato_valido(ficha)
    ]
    avisos_revision.append({
        "modelo": modelo,
        "urgentes omitidos": sum(esperado and not obtenido for esperado, obtenido in validas),
        "avisos innecesarios": sum(not esperado and obtenido for esperado, obtenido in validas),
        "urgentes con ficha inválida": sum(
            esperada["avisar_telegram"] and not formato_valido(ficha)
            for esperada, ficha in zip(ESPERADAS, predicciones, strict=True)
        ),
    })
display(pd.DataFrame(avisos_revision).set_index("modelo"))

rechazadas = revision.loc[~revision["formato válido"]]
if not rechazadas.empty:
    ejemplo = rechazadas.iloc[0]
    print(f"Primera ficha rechazada · {ejemplo['modelo']}")
    print(ejemplo["correo"])
    print("Esperada:", ejemplo["esperada"])
    print("Salida bruta:")
    print(ejemplo["obtenida"])

errores_validos = revision["formato válido"] & ~revision["ficha correcta"]
display(revision.loc[errores_validos, ["modelo", "correo", "esperada", "obtenida"]])

<small>Medidas locales de dos modelos y una política concretos. Fuentes: [SDK Laya 0.3.20](https://pypi.org/project/laya/0.3.20/) · [Pesos Laya](https://huggingface.co/convaiinnovations/laya) · [Qwen 0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) · [Transformers · padding](https://huggingface.co/docs/transformers/main/en/pad_truncation).</small>